# Exercise 2: JIT Compiling Python With Numba

## Learning Objectives

This workbook will be focussed on getting familiar with Numba and using it to Just-In-Time (JIT) compile Python.

Through the exercises, you will:
- Learn how to apply the `@numba.jit` decorator to JIT compile Python functions
- Understand some of the performance differences between NumPy and Numba
- Explore the use of `@numba.jit(parallel=True)` to automatically parallelise NumPy and Numba code

## Setup - Don't Forget To Run!

In [ ]:
import numpy as np
import statistics
import random
import numba
import matplotlib.pyplot as plt

In [ ]:
PROBLEM_SIZE = 10_000_000

## Problem 1: Revisiting Our NumPy Rewrites From Exercise Book 1

Let's start with our simple mean functions:

In [ ]:
def naive_mean(array):
  sum = 0
  for x in array:
    sum += x
  mean = sum / len(array)
  return mean
    
def np_mean(array):
  return np.mean(array)

We applied these functions to both Python Lists, and NumPy Arrays, to see how their performance would compare.

Let's add some `@numba.jit` compiled versions of this function to the comparison:

In [ ]:
naive_mean_jit = numba.jit()(naive_mean)
np_mean_jit = numba.jit()(np_mean)

# Call both once to compile before timing
naive_mean_jit(np.arange(PROBLEM_SIZE))
np_mean_jit(np.arange(PROBLEM_SIZE))

In [ ]:
x_list = list(range(PROBLEM_SIZE))

# Time both functions with Python list
print("Naive Mean on Python List:")
%timeit naive_mean(x_list)
print("JIT'ed Naive Mean on Python List:")
%timeit naive_mean_jit(x_list)
print("NumPy Mean on Python List:")
%timeit np_mean(x_list)
print("JIT'ed NumPy Mean on Python List:")
%timeit np_mean_jit(x_list)

#### As we can see, at best the JIT'ed code is slower, and at worst we get a type inference error!

In [ ]:
x_array = np.array(x_list)

# Time both functions with NumPy Array
print("Naive Mean on NumPy Array:")
%timeit naive_mean(x_array)
print("JIT'ed Naive Mean on NumPy Array:")
%timeit naive_mean_jit(x_array)
print("NumPy Mean on NumPy Array:")
%timeit np_mean(x_array)
print("JIT'ed NumPy Mean on NumPy Array:")
%timeit np_mean_jit(x_array)

#### Now we're getting that NumPy level performance!

#### **Yet another lesson in not using Python Lists for heavy computation!**

### Practical Exercise:

For each of the main exercises from Exercise Book 1 (NumPy), I have pasted in the originaly pure Python code and the example solution NumPy rewrites (please feel to replace with your own rewrites, if they're different).

For each of these code snippets, try applying the `@numba.jit` decorator to them and timing to see if there's any speed up, or slow down.

### Exercise A

In [ ]:
# Pure Python
def norm_to_max_python(x_list):
    max_x = max(x_list)
    norm_to_max_list = [x / max_x for x in x_list]
    return norm_to_max_list

# NumPy Rewrite
def norm_to_max_np(x_array):
    x_array = np.arange(PROBLEM_SIZE)
    norm_to_max_array = x_array / np.max(x_array)
    return norm_to_max_array

In [ ]:
# Your code here, define your JIT'ed functions and run them once to compile them


In [ ]:
# Your timings of the functions here


### Exercise B

In [ ]:
# Pure Python
def strange_sum_python(x_list):
    strange_sum = 0
    for i in range(len(x_list)):
        if x_list[i] > 10:
            strange_sum += x_list[i]**2 + 2
    return strange_sum
    
# NumPy Rewrite
def strange_sum_np(x_array):
    x_array = np.arange(PROBLEM_SIZE)
    strange_sum = np.sum(x_array[x_array > 10]**2+2)
    return strange_sum

In [ ]:
# Your code here, define your JIT'ed functions and run them once to compile them


In [ ]:
# Your timings of the functions here


### Exercise C

In [ ]:
# Pure Python
def dot_product_python(x_list, y_list):
    dot_product = 0
    for i in range(len(x_list)):
        dot_product += x_list[i] * y_list[i]
    return dot_product

# NumPy Rewrite
def dot_product_np(x_array, y_array):
    dot_product = np.dot(x_array, y_array)
    return dot_product

In [ ]:
# Your code here, define your JIT'ed functions and run them once to compile them


In [ ]:
# Your timings of the functions here


## Problem 2: A Closer Look At Numba/JIT

### Measuring The "JIT-Tax"

First, let's see if we can reproduce that `%timeit` warning I showed in the lecture!

In [ ]:
def cheap_function(x):
    return x + 5

cheap_function_jit = numba.jit()(cheap_function)

%timeit -n 1 cheap_function_jit(5) # "-n 1" forces 1 loop per timing run

There it is! But what if we want a bit more detail, how can see the actual timing difference?

For this, we can use `timings = %timeit -o`, and compare `timings.best` to `timings.worst`:

In [ ]:
def cheap_function(x):
    return x + 5

cheap_function_jit = numba.jit()(cheap_function)

timings = %timeit -o -n 1 cheap_function_jit(5) # "-n 1" forces 1 loop per timing run, "-o" stores the timings into an object

print(f"Time with compilation: {timings.worst}")
print(f"Time without compilation: {timings.best}")
print(f"Compilation time: {timings.worst - timings.best}")

Now we get a better idea of the JIT tax, especially with a function this small.

For this function (on my laptop), it looks like compilation is ~70ms. For a small function like this, that's killer!

**That's why we should always meaure, and use JIT only where it'll make a difference!**

### Inspecting Numba's Type Inference

Another way we can probe a bit more deeply into what Numba is doing is by inspecting the types it generates.

We can do this using the `your_function.inspect_types()` function, but we need to compile our function first **(that means we have to call it!)**

In [ ]:
@numba.jit
def types_example(x, y):
    z = x + y
    f = z / 3.0
    return f

types_example(5, 10) # int, int
types_example(3.5, 10) # float, int
types_example(5.6, 3.4) # float, float

types_example.inspect_types()

#### Not only do we get some information on our types, we can also see a bit how Numba handles some of the memory management!

## Problem 3: Applying `parallel=True` To Parallelise Your JIT Functions!

Let's go back to our original, simple mean functions:

In [ ]:
def naive_mean(array):
  sum = 0
  for x in array:
    sum += x
  mean = sum / len(array)
  return mean
    
def np_mean(array):
  return np.mean(array)

We can apply the `parallel=True` flag to both our pure Python mean and our NumPy rewrite! **With a little bit of rewriting...**

Previously, we wrote our `naive_mean` loop with `for x in x_array:`. Whilst we can JIT this, we can only manually parallelise a `numba.prange`, so this will need to be changed!

It looks a little something like this:

In [ ]:
def naive_mean(array):
  sum = 0
  for i in numba.prange(len(array)): # Had to rewrite this from `for x in x_array:`
    sum += array[i] # Also had to change `x` to `array[i]`
  mean = sum / len(array)
  return mean
    
def np_mean(array):
  mean = np.mean(array)
  return mean

naive_mean_jit = numba.jit()(naive_mean)
np_mean_jit = numba.jit()(np_mean)

naive_mean_parallel = numba.jit(parallel=True)(naive_mean)
np_mean_parallel = numba.jit(parallel=True)(np_mean)

# Call both to take care of the JIT tax!
x_array = np.arange(PROBLEM_SIZE)
naive_mean_jit(x_array)
np_mean_jit(x_array)
naive_mean_parallel(x_array)
np_mean_parallel(x_array)

And here's the timings, let's set our thread count to 4:

In [ ]:
x_array = np.arange(PROBLEM_SIZE)

numba.set_num_threads(4)
print(f"Number of threads: {numba.get_num_threads()}")

print("NumPy JIT Serial:")
%timeit np_mean_jit(x_array)
print("NumPy JIT Parallel:")
%timeit np_mean_parallel(x_array)

print("Python JIT Serial:")
%timeit naive_mean_jit(x_array)
print("Python JIT Parallel:")
%timeit naive_mean_parallel(x_array)

Just as we saw for the Monte Carlo Pi calculation in the lectures, the parallel versions of the NumPy and Python implementations perform very similarly!

Let's inspect them in a bit more detail:

In [ ]:
naive_mean_parallel.parallel_diagnostics(level = 1) # Level can be adjusted up to 4 for more information

In [ ]:
np_mean_parallel.parallel_diagnostics(level = 1) # Level can be adjusted up to 4 for more information

Since there's only one loop, it's not too interesting to look at!

At least we can see that both of them have been identified as parallelisable, but we knew that already from the timings.

Let's take a look at a slightly more interesting example:

In [ ]:
def mc_pi(n_samples):
    n_samples_inside = 0
    for i in numba.prange(n_samples):
        x = np.random.random()
        y = np.random.random()
        if x**2 + y**2 <= 1:
            n_samples_inside += 1
    return 4 * n_samples_inside / n_samples

mc_pi_jit = numba.jit()(mc_pi)
mc_pi_jit_par = numba.jit(parallel=True)(mc_pi)

def mc_pi_np(n_samples):
    xs = np.random.random(n_samples)
    ys = np.random.random(n_samples)
    r_sqs = xs**2 + ys**2
    n_samples_inside = np.sum(r_sqs <= 1)
    return 4 * n_samples_inside / n_samples

mc_pi_np_jit = numba.jit()(mc_pi_np)
mc_pi_np_jit_par = numba.jit(parallel=True)(mc_pi_np)

mc_pi_jit(PROBLEM_SIZE) # Run once to activate compilation (remove JIT-Tax from timings)
mc_pi_jit_par(PROBLEM_SIZE) # Run once to activate compilation (remove JIT-Tax from timings)
mc_pi_np_jit(PROBLEM_SIZE) # Run once to activate compilation (remove JIT-Tax from timings)
mc_pi_np_jit_par(PROBLEM_SIZE) # Run once to activate compilation (remove JIT-Tax from timings)

Now let's look at the parallel diagnostics again:

In [ ]:
mc_pi_jit_par.parallel_diagnostics()

In [ ]:
mc_pi_np_jit_par.parallel_diagnostics()

Of course our Python loop was already one parallel loop (that's how we wrote it, only one `prange`!)

What's nice to see is that Numba is able to perform **loop fusion** in order to avoid doing lots of separate parallel loops for our NumPy implementation! These kinds of optimisations make a big difference!

## Parallel Main Exercise

This one is quite simple, it's time to get somre more practice with using `parallel=True`

Go back to the functions you worked on in Problem 2, and see if you can now compile them with `@numba.jit(parallel=True`, and include them in your performance comparisons

## Bonus Challenge: Scaling Plots
### Again, leave this for after doing the next exercise!

As we saw in the lecture, for a given problem size, which implementation runs the fastest out of the ones we have (between NumPy, JIT serial, JIT parallel,...) can vary a lot!

We measured the performance of Monte Carlo Pi estimation over different problem sizes, and compared different implementations, and different thread counts using `numba.set_num_threads()`

The code looked like this (sorry, it's not the neatest!):

In [ ]:
def mc_pi(n_samples):
    n_samples_inside = 0
    for i in numba.prange(n_samples):
        x = np.random.random()
        y = np.random.random()
        if x**2 + y**2 <= 1:
            n_samples_inside += 1
    return 4 * n_samples_inside / n_samples

mc_pi_jit = numba.jit()(mc_pi)
mc_pi_jit_par = numba.jit(parallel=True)(mc_pi)

def mc_pi_np(n_samples):
    xs = np.random.random(n_samples)
    ys = np.random.random(n_samples)
    r_sqs = xs**2 + ys**2
    n_samples_inside = np.sum(r_sqs <= 1)
    return 4 * n_samples_inside / n_samples

mc_pi_np_jit = numba.jit()(mc_pi_np)
mc_pi_np_jit_par = numba.jit(parallel=True)(mc_pi_np)

mc_pi_jit(PROBLEM_SIZE) # Run once to activate compilation (remove JIT-Tax from timings)
mc_pi_jit_par(PROBLEM_SIZE) # Run once to activate compilation (remove JIT-Tax from timings)
mc_pi_np_jit(PROBLEM_SIZE) # Run once to activate compilation (remove JIT-Tax from timings)
mc_pi_np_jit_par(PROBLEM_SIZE) # Run once to activate compilation (remove JIT-Tax from timings)

In [ ]:
funcs = {"NumPy @jit": mc_pi_np_jit, "NumPy @jit(parallel=True)": mc_pi_np_jit_par}

threads_to_test = [1, 2, 4]
problem_sizes = [1, 10, 100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]

numpy_ref = []
for size in problem_sizes:
    time = %timeit -o mc_pi_np(size)
    numpy_ref.append(time.best)


for func_name in funcs.keys():
    if "parallel" in func_name.lower():
        threads = threads_to_test
    else:
        threads = [1]
    for thread in threads:
        numba.set_num_threads(thread)
        
        if thread == 1:
            run_name = f"{func_name} - {thread} Thread"
        else:
            run_name = f"{func_name} - {thread} Threads"

        times=[]
        for size in problem_sizes:
            time = %timeit -o funcs[func_name](size)
            times.append(time.best)

        speedup = []
        for i in range(len(times)):
            speedup.append(numpy_ref[i] / times[i])

        plt.plot(problem_sizes, speedup, label=run_name, ls="-", marker="x")

plt.axhline(y=1.0, color='r', linestyle=':', label="NumPy Performance")
plt.xscale("log")
plt.xlabel("Problem Size")
plt.ylabel("Speedup Over NumPy")
plt.title("JIT Speedup Over NumPy For Different Problem Sizes")
plt.legend()
plt.show()
            

### Your Challenge:

What's quite useful in parallel performance is to check the **strong scaling** of your problem. You can read about this [here](https://hpc-wiki.info/hpc/Scaling).

Your challenge is to:
- Pick 3 problem sizes over the plot above (think which might be interesting based on the plot)
- Modify the above code to plot the **strong scaling** for this problem size (only using the `@numba.jit(parallel=true)` implementation with a varying thread count
- Consider what this means for your performance and optimisation decisions:
  - When is it worth using `parallel=True`
  - Why is it less suitable in other cases?

In [ ]:
# Your code goes here!